# Universe — what is investable, and what is wrong with the data

**Step 2 of the KN Research Process.**

> **Run order: step 3 of 6** (see [`README.md`](../README.md)). After
> `uv run python Data/curator.py`, because this profiles the files it downloaded. Before
> `uv run python Data/refinery.py`, because that joins the security master this writes.

Three questions, in order:

1. **What is in the universe?** `Investable_Universe.csv` carries identity and the grouping this
   strategy cares about. Section 2 fills in everything else from the data provider and writes
   `Security_Master.csv`.
2. **What is wrong with what we downloaded?** Section 3 reads the Curator's files back and writes
   `Data_Issues.csv`, so the problems are known *before* a portfolio is built on them.
3. **When does each asset actually become usable?** Section 4. A feature with a five-year training
   window does not start on the first row of the file, and a backtest that assumes it does is
   measuring a shorter history than it thinks.

**Eligibility is decided here and nowhere else.** Everything downstream reads the security master
and does not second-guess it.

## Where this sits

```
Universe/Investable_Universe.csv   the seed, committed  -->  you edit this to change the universe
        |
        +--> Data/curator.py       downloads one file per ticker in it
        |
        +--> this notebook         Security_Master.csv, Data_Issues.csv, Charts/
```

## Swapping the universe

The seed is the only thing that decides what this repository is about. Replace its rows with
crypto pairs, FX crosses or single stocks and every stage below still runs — nothing downstream
names an asset class. Two columns are required (`ticker`, `name`); the rest are yours. This one
carries `asset_class` and `asset_group` because the example strategy compares asset classes, and
`index_name` / `index_ticker` because each ETF is a *tradable proxy* for an index the source paper
backtested, and that distinction is worth keeping visible.

To swap in a different file entirely, point `RAW_UNIVERSE_PATH` below and
`INVESTABLE_UNIVERSE_PATH` in `Data/curator.py` at it. Those are the only two places that name it.

---

## 0 · Setup

Paths and credentials. Nothing here touches the network.

In [ ]:
"""Step 2 - Universe. Security master, data issues, and when each asset becomes usable."""
import collections
import concurrent.futures
import json
import os
import pathlib
import urllib.error
import urllib.parse
import urllib.request

import dotenv
import matplotlib.pyplot
import pandas


def find_repo_root(start):
    """Walk up from `start` to the directory holding pyproject.toml and Universe/."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "Universe").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(pathlib.Path.cwd())
UNIVERSE_DIR = REPO_ROOT / "Universe"
CHART_DIR = UNIVERSE_DIR / "Charts"
PROVIDER_CACHE_DIR = UNIVERSE_DIR / "Provider_Cache"
CURATOR_DIR = REPO_ROOT / "Data" / "Curator" / "Time_Series"

RAW_UNIVERSE_PATH = UNIVERSE_DIR / "Investable_Universe.csv"
SECURITY_MASTER_PATH = UNIVERSE_DIR / "Security_Master.csv"
DATA_ISSUES_PATH = UNIVERSE_DIR / "Data_Issues.csv"

for directory in (CHART_DIR, PROVIDER_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Downloaded alongside the universe because the engine prices every ticker from one directory.
# They are not part of the cross-section, so every count below excludes them.
NON_UNIVERSE_TICKERS = frozenset({"AOR", "BIL", "SPY"})

# The training window the Refinery's regime model needs before it will label a day. Section 4
# turns it into a first-usable date per asset, which is the real start of the backtest.
SIGNAL_WARMUP_DAYS = 1260

DATA_PROVIDER = "financial_modeling_prep"
PROVIDER_TIMEOUT_SECONDS = 30

dotenv.load_dotenv(REPO_ROOT / "Config" / ".env")
PROVIDER_API_KEYS = {"financial_modeling_prep": os.getenv("KNDC_API_KEY_FMP")}

print(f"Repo root       : {REPO_ROOT}")
print(f"Seed            : {RAW_UNIVERSE_PATH.relative_to(REPO_ROOT)}")
print(f"Curator data    : {CURATOR_DIR.relative_to(REPO_ROOT)}"
      f" ({len(list(CURATOR_DIR.glob('*.csv')))} file(s))")
print(f"Provider        : {DATA_PROVIDER}"
      f" (key {'loaded' if PROVIDER_API_KEYS.get(DATA_PROVIDER) else 'MISSING'})")

---

## 1 · The seed

Two things to establish before anything downstream trusts this file.

1. **Every ticker is unique, and so is every ISIN.** A repeated ISIN under two tickers is a
   *ticker change* — the same security renamed — and the two legs have to be stitched into one
   position or the backtest holds it twice. A point-in-time universe is supposed to contain
   these; what it must not do is hide them.
2. **The grouping is complete.** Whatever column the strategy compares things by, a missing value
   in it is a security that will silently drop out of every group-level view.

In [ ]:
# utf-8-sig strips the byte-order mark, otherwise the first column is named "﻿ticker".
seed = pandas.read_csv(RAW_UNIVERSE_PATH, encoding="utf-8-sig", dtype=str)
print(f"{RAW_UNIVERSE_PATH.name}: {seed.shape[0]} rows x {seed.shape[1]} columns\n")
print(seed.to_string(index=False))

REQUIRED_COLUMNS = ("ticker", "name")
missing_required = [column for column in REQUIRED_COLUMNS if column not in seed.columns]
assert not missing_required, f"seed is missing required column(s): {missing_required}"

duplicate_tickers = seed.loc[seed["ticker"].duplicated(keep=False), "ticker"].tolist()
shared_isins = (
    seed[seed["isin"].notna() & seed["isin"].duplicated(keep=False)]
    .groupby("isin")["ticker"].apply(list).to_dict()
    if "isin" in seed.columns else {}
)

print(f"\nduplicate tickers : {duplicate_tickers or 'none'}")
print(f"shared ISINs      : {shared_isins or 'none - no ticker changes to stitch'}")
for column in seed.columns:
    blanks = int(seed[column].isna().sum())
    if blanks:
        print(f"  {column}: {blanks} blank value(s)")

UNIVERSE_TICKERS = tuple(seed["ticker"].dropna().unique())
print(f"\nUNIVERSE_TICKERS ready ({len(UNIVERSE_TICKERS)})")

---

## 2 · Enrichment — build the security master

The seed gives identity and grouping. Everything else comes from the provider.

**Two layers, on purpose.** The network layer caches the provider's *raw* payload to
`Universe/Provider_Cache/` as JSON Lines; the shaping layer derives `Security_Master.csv` from
that cache. So re-running this notebook costs nothing, changing the column mapping never triggers
a refetch, and the untouched payload stays available for fields this notebook does not yet use.

**Adding a provider** is one fetch function plus one normaliser in `PROVIDER_ADAPTERS`. Nothing
else changes.

In [ ]:
FINANCIAL_MODELING_PREP_PROFILE_URL = "https://financialmodelingprep.com/stable/profile"

# What a provider adds on top of the seed. Kept short: a column nobody reads is a column that
# quietly goes stale.
PROVIDER_COLUMNS = (
    "provider_name", "category", "active", "exchange",
    "currency", "provider_isin", "inception_date",
)


def fetch_profile_financial_modeling_prep(ticker, api_key):
    """
    Return FMP's profile payload for `ticker`, or None when it has no profile.

    One request per ticker: the endpoint takes a single symbol, and a comma-separated list comes
    back as an empty array rather than several records.
    """
    query = urllib.parse.urlencode({"symbol": ticker, "apikey": api_key})
    request = urllib.request.Request(f"{FINANCIAL_MODELING_PREP_PROFILE_URL}?{query}")
    with urllib.request.urlopen(request, timeout=PROVIDER_TIMEOUT_SECONDS) as response:
        payload = json.load(response)
    return payload[0] if payload else None


def normalise_profile_financial_modeling_prep(payload):
    """Map one FMP profile payload onto the master schema."""
    # FMP exposes ETF / fund / ADR as three booleans rather than one category string.
    category = "Stock"
    for flag, label in (("isEtf", "ETF"), ("isFund", "Fund"), ("isAdr", "ADR")):
        if payload.get(flag):
            category = label
    return {
        "provider_name": payload.get("companyName"),
        "category": category,
        "active": payload.get("isActivelyTrading"),
        "exchange": payload.get("exchange"),
        "currency": payload.get("currency"),
        "provider_isin": payload.get("isin"),
        "inception_date": payload.get("ipoDate"),
    }


ProviderAdapter = collections.namedtuple("ProviderAdapter", ("fetch", "normalise"))
PROVIDER_ADAPTERS = {
    "financial_modeling_prep": ProviderAdapter(
        fetch=fetch_profile_financial_modeling_prep,
        normalise=normalise_profile_financial_modeling_prep,
    ),
}
ADAPTER = PROVIDER_ADAPTERS[DATA_PROVIDER]

RUN_ENRICHMENT = True   # False -> no network calls; report what the cache already holds
REFETCH_ALL = False     # True  -> ignore the cache and refetch every ticker
PROFILE_CACHE_PATH = PROVIDER_CACHE_DIR / f"{DATA_PROVIDER}_profiles.jsonl"


def load_profile_cache(path):
    """Read the JSON Lines payload cache into {ticker: payload-or-None}."""
    if not path.is_file():
        return {}
    cached = {}
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                cached[record["ticker"]] = record["payload"]
    return cached


def fetch_one_profile(ticker):
    """Fetch `ticker` and return (ticker, payload, outcome). Never raises."""
    try:
        payload = ADAPTER.fetch(ticker, PROVIDER_API_KEYS[DATA_PROVIDER])
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError, OSError) as error:
        return ticker, None, f"failed: {type(error).__name__}"
    return ticker, payload, "fetched" if payload else "no profile"


profile_cache = {} if REFETCH_ALL else load_profile_cache(PROFILE_CACHE_PATH)
pending = [ticker for ticker in UNIVERSE_TICKERS if ticker not in profile_cache]
print(f"Cache: {len(profile_cache)} payload(s) on disk, {len(pending)} pending.")

if RUN_ENRICHMENT and pending:
    assert PROVIDER_API_KEYS.get(DATA_PROVIDER), f"no API key for {DATA_PROVIDER}; see Config/.env"
    if REFETCH_ALL and PROFILE_CACHE_PATH.is_file():
        PROFILE_CACHE_PATH.unlink()
    # Appending rather than rewriting is what makes an interrupted run resumable.
    with (
        concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor,
        PROFILE_CACHE_PATH.open("a", encoding="utf-8") as handle,
    ):
        for ticker, payload, outcome in executor.map(fetch_one_profile, pending):
            handle.write(json.dumps({"ticker": ticker, "payload": payload}) + "\n")
            print(f"  {ticker}: {outcome}")
    profile_cache = load_profile_cache(PROFILE_CACHE_PATH)
elif not RUN_ENRICHMENT:
    print("RUN_ENRICHMENT = False - no network calls.")

MISSING_PROFILES = tuple(t for t in UNIVERSE_TICKERS if not profile_cache.get(t))
print(f"\n{len(UNIVERSE_TICKERS) - len(MISSING_PROFILES)}/{len(UNIVERSE_TICKERS)}"
      f" tickers have a profile. Missing: {MISSING_PROFILES or 'none'}")

### 2.1 · Shape the master and reconcile against the seed

**The seed wins.** Its `ticker` and `isin` define the universe, so a provider value never
overwrites them — it is compared instead. An ISIN disagreement means the provider now points that
symbol at a *different security*, which is a recycled ticker, and joining on ticker alone across
the whole history would silently mix two companies.

In [ ]:
rows = []
for ticker in UNIVERSE_TICKERS:
    row = {"ticker": ticker}
    payload = profile_cache.get(ticker)
    row.update(dict.fromkeys(PROVIDER_COLUMNS))
    if payload:
        row.update(ADAPTER.normalise(payload))
    rows.append(row)

security_master = seed.merge(pandas.DataFrame(rows), on="ticker", how="left")
security_master["active"] = security_master["active"].astype("boolean")
security_master["inception_date"] = pandas.to_datetime(
    security_master["inception_date"], errors="coerce"
)

if "isin" in security_master.columns:
    conflicts = security_master[
        security_master["provider_isin"].notna()
        & (security_master["provider_isin"] != security_master["isin"])
    ]
    print(f"ISIN disagreements (seed vs provider): {len(conflicts)}")
    if len(conflicts):
        print(conflicts[["ticker", "name", "isin", "provider_name", "provider_isin"]]
              .to_string(index=False))
        print("  -> the provider points this symbol at a different security. The seed wins.")

security_master.to_csv(SECURITY_MASTER_PATH, index=False)
print(f"\nWritten: {SECURITY_MASTER_PATH.relative_to(REPO_ROOT)}"
      f" ({security_master.shape[0]} rows x {security_master.shape[1]} columns)")

population = pandas.DataFrame({
    "non_empty": security_master.notna().sum(),
    "share": (security_master.notna().sum() / len(security_master)).map("{:.0%}".format),
    "source": ["seed" if column in seed.columns else DATA_PROVIDER
               for column in security_master.columns],
})
print("\nColumn population:")
print(population.to_string())

---

## 3 · Composition — what the universe is made of, and how long it has existed

Two facts decide what a backtest can honestly claim. **What the universe is made of** sets what
diversification is even available; **when each member started trading** sets the window, because a
universe is only complete from the inception of its youngest member.

In [ ]:
INK = "#0b0b0b"
MUTED = "#52514e"
BLUE = "#2a78d6"
ORANGE = "#eb6834"
SURFACE = "#fcfcfb"
GRID = "#ebeae5"

matplotlib.pyplot.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": "#d8d7d2",
    "axes.labelcolor": MUTED, "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 12, "figure.dpi": 110,
    "savefig.dpi": 160, "savefig.bbox": "tight",
})


def style_axes(axes, title=None, subtitle=None, xlabel=None, ylabel=None):
    """Left-aligned bold title, optional subtitle line, recessive grid, no top/right spines."""
    if title:
        axes.set_title(title, loc="left", pad=24 if subtitle else 8, weight="bold")
    if subtitle:
        axes.annotate(subtitle, xy=(0, 1), xycoords="axes fraction", xytext=(0, 6),
                      textcoords="offset points", fontsize=9, color=MUTED, va="bottom", ha="left")
    if xlabel:
        axes.set_xlabel(xlabel)
    if ylabel:
        axes.set_ylabel(ylabel)
    axes.grid(True, color=GRID, linewidth=0.8)
    axes.set_axisbelow(True)
    for side in ("top", "right"):
        axes.spines[side].set_visible(False)
    return axes


def save(figure, file_name):
    """Write a figure to Universe/Charts/ and show it."""
    figure.tight_layout()
    figure.savefig(CHART_DIR / file_name)
    matplotlib.pyplot.show()


print("Chart helpers ready.")

In [ ]:
GROUP_COLUMN = "asset_group"
CLASS_COLUMN = "asset_class"

composition = security_master.groupby(GROUP_COLUMN)["ticker"].apply(list)
print("Universe by group:")
for group, tickers in composition.items():
    print(f"  {group:<14} {len(tickers):>2}  {', '.join(tickers)}")

inceptions = security_master.dropna(subset=["inception_date"]).sort_values("inception_date")
complete_from = inceptions["inception_date"].max()
print(f"\nYoungest member : {inceptions.iloc[-1]['ticker']}"
      f" ({inceptions['inception_date'].max().date()})")
print(f"Universe is complete only from {complete_from.date()} onward.")

figure, axes = matplotlib.pyplot.subplots(figsize=(10, 4.6))
positions = list(range(len(inceptions)))
axes.scatter(inceptions["inception_date"], positions, color=BLUE, s=42, zorder=3)
axes.axvline(complete_from, color=ORANGE, linewidth=1.6, linestyle="--")
axes.set_yticks(positions)
axes.set_yticklabels(
    [f"{row.ticker}  ({row.asset_class})" for row in inceptions.itertuples()]
)
style_axes(
    axes, "When each asset started trading",
    subtitle=f"the dashed line is {complete_from.date()}, the first date the universe is complete",
    xlabel="year",
)
axes.spines["left"].set_visible(False)
save(figure, "universe_inception_dates.png")

---

## 4 · Data-issues register — what the downloaded files actually contain

The seed says what *should* exist. This section reads what *does*, and writes
`Universe/Data_Issues.csv`. Three checks, each of which has cost somebody real time somewhere:

| Check | The failure it catches |
| --- | --- |
| **Missing file** | a ticker in the seed the provider does not carry, which becomes a silent hole in the panel |
| **Schema drift** | a directory holding two column sets, which makes every downstream read conditional |
| **History shape** | a series that starts late or stops early, which shortens the backtest without saying so |

In [ ]:
curator_files = {path.stem: path for path in sorted(CURATOR_DIR.glob("*.csv"))}
assert curator_files, f"no files in {CURATOR_DIR} - run: uv run python Data/curator.py"

universe_files = {t: p for t, p in curator_files.items() if t not in NON_UNIVERSE_TICKERS}
schemas = collections.Counter()
profiles = []
for ticker, path in universe_files.items():
    header = tuple(pandas.read_csv(path, nrows=0).columns)
    schemas[header] += 1
    dates = pandas.read_csv(path, usecols=["m_date"], parse_dates=["m_date"])["m_date"]
    profiles.append({
        "ticker": ticker, "rows": len(dates),
        "first_date": dates.min(), "last_date": dates.max(),
        "columns": len(header),
    })

history = pandas.DataFrame(profiles).sort_values("first_date")
majority_schema, majority_count = schemas.most_common(1)[0]

print(f"Universe files : {len(universe_files)} of {len(UNIVERSE_TICKERS)} seed tickers")
print(f"Extra files    : {sorted(set(curator_files) - set(universe_files))}"
      " (cash proxy and benchmarks - priced by the engine, excluded from the cross-section)")
print(f"Schemas        : {len(schemas)}"
      f" ({'consistent' if len(schemas) == 1 else 'MIXED - see below'});"
      f" {majority_count} file(s) carry the majority schema of {len(majority_schema)} columns")
print()
print(history.assign(
    first_date=lambda f: f["first_date"].dt.date,
    last_date=lambda f: f["last_date"].dt.date,
).to_string(index=False))

In [ ]:
LATE_START_TOLERANCE_DAYS = 5
EARLY_END_TOLERANCE_DAYS = 5

panel_first = history["first_date"].min()
panel_last = history["last_date"].max()

issues = []
for ticker in UNIVERSE_TICKERS:
    if ticker not in universe_files:
        issues.append({
            "ticker": ticker, "issue": "no_data_file",
            "detail": "the provider returned nothing for this symbol",
            "effect": "absent from the panel; every cross-sectional count is one smaller",
        })

for row in history.itertuples():
    header = tuple(pandas.read_csv(universe_files[row.ticker], nrows=0).columns)
    if header != majority_schema:
        issues.append({
            "ticker": row.ticker, "issue": "schema_drift",
            "detail": f"{len(header)} columns against the majority {len(majority_schema)}",
            "effect": "downstream reads of this file need a special case",
        })
    if (row.first_date - panel_first).days > LATE_START_TOLERANCE_DAYS:
        issues.append({
            "ticker": row.ticker, "issue": "late_start",
            "detail": f"first row {row.first_date.date()}, panel starts {panel_first.date()}",
            "effect": "the cross-section is smaller before this date",
        })
    if (panel_last - row.last_date).days > EARLY_END_TOLERANCE_DAYS:
        issues.append({
            "ticker": row.ticker, "issue": "early_end",
            "detail": f"last row {row.last_date.date()}, panel ends {panel_last.date()}",
            "effect": "delisted or halted; a held position must be exited on its last priced day",
        })

data_issues = pandas.DataFrame(
    issues, columns=["ticker", "issue", "detail", "effect"]
).sort_values(["issue", "ticker"])
data_issues.to_csv(DATA_ISSUES_PATH, index=False)

print(f"Written: {DATA_ISSUES_PATH.relative_to(REPO_ROOT)} ({len(data_issues)} issue(s))")
if len(data_issues):
    print()
    print(data_issues.to_string(index=False))
else:
    print("No issues found. That is worth being suspicious of on a larger universe.")

---

## 5 · When the universe is actually usable

A file that starts in 2010 does not give a signal in 2010. Every feature has a warm-up, and the
regime model in the Refinery needs `SIGNAL_WARMUP_DAYS` of history before it will label anything.

**The date that matters is the last one in the right-hand column below** — the first day on which
every asset can be both priced *and* labelled. Before it the strategy is choosing from a smaller
menu than it appears to be, and a backtest that starts earlier is quietly comparing books drawn
from different universes.

In [ ]:
usable = history.copy()
usable["first_signal_date"] = [
    pandas.read_csv(universe_files[row.ticker], usecols=["m_date"], parse_dates=["m_date"])
    ["m_date"].iloc[SIGNAL_WARMUP_DAYS]
    if row.rows > SIGNAL_WARMUP_DAYS else pandas.NaT
    for row in history.itertuples()
]
usable["warmup_years"] = SIGNAL_WARMUP_DAYS / 252

FULL_UNIVERSE_DATE = usable["first_signal_date"].max()
print(usable[["ticker", "rows", "first_date", "first_signal_date"]].assign(
    first_date=lambda f: f["first_date"].dt.date,
    first_signal_date=lambda f: f["first_signal_date"].dt.date,
).to_string(index=False))
print(f"\nPriced from        : {panel_first.date()}")
print(f"First signal       : {usable['first_signal_date'].min().date()}")
print(f"ALL assets signalled: {FULL_UNIVERSE_DATE.date()}"
      f"  <- the honest start of a backtest over the whole universe")

figure, axes = matplotlib.pyplot.subplots(figsize=(11, 4.6))
ordered = usable.sort_values("first_date")
for position, row in enumerate(ordered.itertuples()):
    axes.plot([row.first_date, row.last_date], [position, position],
              color=GRID, linewidth=6, solid_capstyle="butt")
    if pandas.notna(row.first_signal_date):
        axes.plot([row.first_signal_date, row.last_date], [position, position],
                  color=BLUE, linewidth=6, solid_capstyle="butt")
axes.axvline(FULL_UNIVERSE_DATE, color=ORANGE, linewidth=1.6, linestyle="--")
axes.set_yticks(range(len(ordered)))
axes.set_yticklabels(ordered["ticker"])
style_axes(
    axes, "Priced (grey) against signalled (blue)",
    subtitle="the dashed line is the first date every asset has a regime label",
)
axes.spines["left"].set_visible(False)
save(figure, "universe_usable_history.png")

---

## 6 · Handoff

In [ ]:
summary = pandas.DataFrame({
    "metric": [
        "seed tickers", "with a data file", "asset groups",
        "priced from", "universe complete from", "all assets signalled from",
        "data issues", "security master", "issues register",
    ],
    "value": [
        f"{len(UNIVERSE_TICKERS)}",
        f"{len(universe_files)}",
        f"{security_master[GROUP_COLUMN].nunique()}",
        f"{panel_first.date()}",
        f"{complete_from.date()}",
        f"{FULL_UNIVERSE_DATE.date()}",
        f"{len(data_issues)}",
        str(SECURITY_MASTER_PATH.relative_to(REPO_ROOT)),
        str(DATA_ISSUES_PATH.relative_to(REPO_ROOT)),
    ],
})
print(summary.to_string(index=False))
print(f"\nCharts: {CHART_DIR.relative_to(REPO_ROOT)}")

| Output | Consumed by |
| --- | --- |
| `Universe/Security_Master.csv` | `Data/refinery.py`, which joins its columns onto the panel |
| `Universe/Data_Issues.csv` | the caveats section of every `FINDINGS_N.md` |
| `Universe/Charts/` | `FINDINGS_N.md` |

## Open items for the Data stage

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **Classification is a snapshot, not a history.** The provider returns what a security is *today*. For this universe that is harmless — an S&P 500 ETF has always been one — but on a stock universe a name reclassified mid-window is misattributed before its move. The `*_current` suffix in the Refinery marks exactly this. |
| 2 | **The last day of a delisted name is unaudited.** Nothing here checks whether a truncated series ends on a real final price or on a provider gap. On a universe that retains delisted names, the missing returns are disproportionately the bad ones. |
| 3 | **Inception dates come from the provider, not from the price file.** The two agree here; where they disagree, the price file is what the backtest actually trades and the master is what a reader believes. |